# 5x5 V2V grid demo — Wan 2.1 VACE 1.3B

Builds a single tiled mp4 showing the model's editing ability across themed prompts.

- **Rows** = ground-truth Waymo dashcam clips.
- **Column 0** = ground truth (no inference).
- **Columns 1-4** = the same clip restyled per a Chinese themed prompt: `snow` / `mario_kart` / `cyberpunk` / `desert`.
- **Output** (`outputs/v2v_recon/<stamp>_grid_demo/`):
  - `grid.mp4` — composed labeled grid.
  - `<row_label>/input.mp4` + `<row_label>/<row_label>_<preset>.mp4` — one folder per input clip with the GT symlink and 4 themed edits inside.
  - `meta.json` — full prompts + cache keys + params + timing.
- Per-cell mp4s are also cached at `outputs/v2v_recon/_cache/cells/<sha1>.mp4` and reused across runs.

**Multi-GPU.** Every id in `GPU_IDS` is utilized — one Wan VACE pipeline is loaded per GPU and cells are dispatched in parallel via a thread pool. With `[3, 5]` (two A6000s), wall time is ~half the single-GPU run.

**Editing knobs.**
- `GUIDANCE = 7.5` — high CFG for stronger prompt steering.
- `CONDITIONING_SCALE` — by default each preset uses its own tuned value (snow=0.5, mario_kart=0.75, cyberpunk=0.6, desert=0.4). The optimal balance depends on how transformative the edit is: scene replacement (desert) needs low cond; style-only redraw (mario_kart) needs high cond.
- Prompts in Chinese (Wan 2.1 was trained primarily on Chinese text-video pairs).

All heavy lifting is in `inference/utils/runner.py:run_grid`. The notebook is just parameters + one call + preview.

## Parameters

Smoke test (~30s on one GPU): set `VIDEO_IDS=["3"]`, `PRESET_NAMES=["snow"]`, `STEPS=4`.

In [ ]:
VIDEO_IDS = ["0","55410","5410","8888"]               # rows, 8888, 55410
PRESET_NAMES = ["snow", "mario_kart", "cyberpunk", "desert"]  # columns 1-4 (col 0 is GT)
GPU_IDS = [0,1,2,3, 5]                                       # all listed GPUs are used in parallel

STEPS = 50                  # full quality
GUIDANCE = 8.0              # CFG; higher = stronger prompt steering
SEED = 0

# Per-preset conditioning_scale (VACE control strength).
#   Higher = source dominates (composition preserved, less editing).
#   Lower  = prompt dominates (scene replaced more, may hallucinate).
# Defaults are tuned per theme in prompts.py:
#   snow=0.5  mario_kart=0.75  cyberpunk=0.6  desert=0.4
# Set to None to use these per-preset defaults. Use a float to force uniform.
# Use a dict to override selectively.
CONDITIONING_SCALE = 0.4
# Examples to try:
# CONDITIONING_SCALE = 0.55                                # force uniform
# CONDITIONING_SCALE = {"mario_kart": 0.8, "desert": 0.35} # selective override

## Run

Cold runtime ≈ `(20 inferences × ~3 min/cell) / len(GPU_IDS)` (so ~30 min on two GPUs at `STEPS=50`). Cached re-runs finish in seconds.

In [ ]:
import sys
from pathlib import Path

INFERENCE_DIR = Path('/home/alec/ARV2V/inference')
if str(INFERENCE_DIR) not in sys.path:
    sys.path.insert(0, str(INFERENCE_DIR))

from utils import run_grid

result = run_grid(
    video_ids=VIDEO_IDS,
    preset_names=PRESET_NAMES,
    gpu_ids=GPU_IDS,
    steps=STEPS,
    guidance=GUIDANCE,
    conditioning_scale=CONDITIONING_SCALE,
    seed=SEED,
)

## Preview

In [ ]:
from IPython.display import Video, display

print(result['grid_path'])
display(Video(str(result['grid_path']), embed=True))